In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import copy
import random
import math

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans

from datetime import datetime

In [ ]:
dbConnectionString = "sqlite:///baseball_info.db"

engine = create_engine(dbConnectionString)

In [ ]:
timeSeriesHittingQuery = "SELECT SeasonStatsHitting.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2015 and 2025 and season != 2020 ORDER BY SeasonStatsHitting.playerId,season"

df = pd.read_sql(timeSeriesHittingQuery, engine)

In [ ]:
hasMultipleSeasons = df["playerId"].value_counts() > 1
df = df[df["playerId"].isin(hasMultipleSeasons[hasMultipleSeasons].index)]

In [ ]:
df.drop(df[df['plateAppearances'] == 0].index, inplace=True)

In [ ]:
def calculate_age_from_timestamps(birth_timestamp, current_timestamp):
    birth_date   = datetime.fromtimestamp(birth_timestamp)
    current_date = datetime.fromtimestamp(current_timestamp)

    age = current_date.year - birth_date.year

    # Adjust age if the birthday hasn't occurred yet in the current year
    if (current_date.month, current_date.day) < (birth_date.month, birth_date.day):
        age -= 1
    
    return age

In [ ]:
twentyFifteenDateString     = "2015-04-05 00:00:00"
twentySixteenDateString     = "2016-04-03 00:00:00"
twentySeventeenDateString   = "2017-04-02 00:00:00"
twentyEighteenDateString    = "2018-03-29 00:00:00"
twentyNineteenDateString    = "2019-03-20 00:00:00"
twentyTwentyOneDateString   = "2021-04-01 00:00:00"
twentyTwentyTwoDateString   = "2022-04-07 00:00:00"
twentyTwentyThreeDateString = "2023-03-30 00:00:00"
twentyTwentyFourDateString  = "2024-03-20 00:00:00"
twentyTwentyFiveDateString  = "2025-03-18 00:00:00"

# Parse the string into a datetime object
twentyFifteenDateString     = datetime.strptime(twentyFifteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySixteenDateString     = datetime.strptime(twentySixteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySeventeenDateString   = datetime.strptime(twentySeventeenDateString   , "%Y-%m-%d %H:%M:%S")
twentyEighteenDateString    = datetime.strptime(twentyEighteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyNineteenDateString    = datetime.strptime(twentyNineteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyTwentyOneDateString   = datetime.strptime(twentyTwentyOneDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyTwoDateString   = datetime.strptime(twentyTwentyTwoDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyThreeDateString = datetime.strptime(twentyTwentyThreeDateString , "%Y-%m-%d %H:%M:%S")
twentyTwentyFourDateString  = datetime.strptime(twentyTwentyFourDateString  , "%Y-%m-%d %H:%M:%S")
twentyTwentyFiveDateString  = datetime.strptime(twentyTwentyFiveDateString  , "%Y-%m-%d %H:%M:%S")

# Convert to Unix timestamp
twentyFifteenTimeStamp     = int(twentyFifteenDateString     .timestamp())
twentySixteenTimeStamp     = int(twentySixteenDateString     .timestamp())
twentySeventeenTimeStamp   = int(twentySeventeenDateString   .timestamp())
twentyEighteenTimeStamp    = int(twentyEighteenDateString    .timestamp())
twentyNineteenTimeStamp    = int(twentyNineteenDateString    .timestamp())
twentyTwentyOneTimeStamp   = int(twentyTwentyOneDateString   .timestamp())
twentyTwentyTwoTimeStamp   = int(twentyTwentyTwoDateString   .timestamp())
twentyTwentyThreeTimeStamp = int(twentyTwentyThreeDateString .timestamp())
twentyTwentyFourTimeStamp  = int(twentyTwentyFourDateString  .timestamp())
twentyTwentyFiveTimeStamp  = int(twentyTwentyFiveDateString  .timestamp())

seasonToStartTimeStamp = {}

seasonToStartTimeStamp[2015] = twentyFifteenTimeStamp   
seasonToStartTimeStamp[2016] = twentySixteenTimeStamp   
seasonToStartTimeStamp[2017] = twentySeventeenTimeStamp   
seasonToStartTimeStamp[2018] = twentyEighteenTimeStamp   
seasonToStartTimeStamp[2019] = twentyNineteenTimeStamp   
seasonToStartTimeStamp[2021] = twentyTwentyOneTimeStamp  
seasonToStartTimeStamp[2022] = twentyTwentyTwoTimeStamp  
seasonToStartTimeStamp[2023] = twentyTwentyThreeTimeStamp
seasonToStartTimeStamp[2024] = twentyTwentyFourTimeStamp 
seasonToStartTimeStamp[2025] = twentyTwentyFiveTimeStamp

In [ ]:
df['dob'] = df.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

In [ ]:
def getWindowedFeaturesAndLabels(frame, maxWindowSize):
    frameArray = frame.values.tolist()

    inputs = []
    labels = []
    
    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[5:6] + labelTest[10:12] + labelTest[18:19] + labelTest[23:24]

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [0] * 34
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    inputs.append(copy.deepcopy(windowArray))
                    labels.append(copy.deepcopy(relevantLabels))

                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    return inputs,labels

In [ ]:
features,labels = getWindowedFeaturesAndLabels(df, 4)

In [ ]:
fullFeaturesArray = np.array(features)
fullLabelsArray   = np.array(labels)

In [ ]:
fullLabelsArray[1]

In [ ]:
runBins              = np.linspace(min(fullLabelsArray[:,0]), max(fullLabelsArray[:,0]), num=4)
homeRunsBins         = np.linspace(min(fullLabelsArray[:,1]), max(fullLabelsArray[:,1]), num=4)
rbiBins              = np.linspace(min(fullLabelsArray[:,2]), max(fullLabelsArray[:,2]), num=4)
stolenBaseBins       = np.linspace(min(fullLabelsArray[:,3]), max(fullLabelsArray[:,3]), num=4)
onBasePercentageBins = np.linspace(min(fullLabelsArray[:,4]), max(fullLabelsArray[:,4]), num=4)

runsBinned             = np.digitize(fullLabelsArray[:,0], runBins, right=True)
homeRunsBinned         = np.digitize(fullLabelsArray[:,1], homeRunsBins, right=True)
rbisBinned             = np.digitize(fullLabelsArray[:,2], rbiBins, right=True)
stolenBasesBinned      = np.digitize(fullLabelsArray[:,3], stolenBaseBins, right=True)
onBasePercentageBinned = np.digitize(fullLabelsArray[:,4], onBasePercentageBins, right=True)

fullLabelsArrayBinned = np.column_stack((runsBinned, homeRunsBinned, rbisBinned, stolenBasesBinned, onBasePercentageBinned))

In [ ]:
fullLabelsArrayBinned.shape

In [ ]:
fullFeaturesArray.shape

In [ ]:
binCount = {}

for label in fullLabelsArrayBinned:
    labelString = str(label[0]) + str(label[1]) + str(label[2]) + str(label[3]) + str(label[4])

    if labelString in binCount:
        binCount[labelString] = binCount[labelString] + 1
    else:
        binCount[labelString] = 1

In [ ]:
sortedBinCount = dict(sorted(binCount.items(), key=lambda item: item[1], reverse=True))

In [ ]:
len(sortedBinCount)

In [ ]:
binOne   = {}
binTwo   = {}
binThree = {}
binFour  = {}

index = 1

for key in sortedBinCount:
    if index <= 22:
        binOne[key] = sortedBinCount[key]
    elif index > 22 and index <= 44:
        binTwo[key] = sortedBinCount[key]
    elif index > 44 and index <= 66:
        binThree[key] = sortedBinCount[key]
    else:
        binFour[key] = sortedBinCount[key]

    index += 1

In [ ]:
labels = list(sortedBinCount.keys())
count  = list(sortedBinCount.values())

plt.figure(figsize=(16, 12))
bar = plt.bar(labels, count)

_ = plt.xticks(labels, rotation="vertical")
_ = plt.bar_label(bar, count)

In [ ]:
condition = (fullLabelsArrayBinned[:, 0] == 1) & (fullLabelsArrayBinned[:, 1] == 0) & (fullLabelsArrayBinned[:, 2] == 1) & (fullLabelsArrayBinned[:, 3] == 0) & (fullLabelsArrayBinned[:, 4] == 0)

matchingLabelIndices = np.where(condition)[0]

len(matchingLabelIndices)

In [ ]:
print(labels)
binIndexes = []

for label in labels:
    labelNumber = int(label)
    
    labelArray = []
    
    while labelNumber > 0:
        digit = labelNumber % 10
        
        labelArray.insert(0, digit)
        
        labelNumber = int(labelNumber / 10)
        
    while len(labelArray) < 5:
        labelArray.insert(0, 0)
        
    binIndexes.append(labelArray)
    
print(binIndexes)

In [ ]:
def addNoiseToPoints(point, label):
    noisyPoint = np.copy(point)
    
    mean               = 0
    stdDevWholeNumbers = 1
    stdDevPercentages  = 0.01
    
    wholeNumbersMask = np.zeros_like(noisyPoint, dtype=bool)
    
    allZerosForSecondWindow = (noisyPoint[32:64] == 0.0).all()
    allZerosForThirdWindow  = (noisyPoint[64:  ] == 0.0).all()
    
    wholeNumbersMask[:20  ] = True
    
    #print(allZerosForSecondWindow)
    #print(allZerosForThirdWindow)
    
    if allZerosForSecondWindow != True:
        wholeNumbersMask[32:52] = True
        
    if allZerosForThirdWindow != True:
        wholeNumbersMask[64:84] = True
    
    percentageNumbersMask = np.zeros_like(noisyPoint, dtype=bool)
    
    percentageNumbersMask[20:32] = True
    
    if allZerosForSecondWindow != True:
        percentageNumbersMask[52:64] = True
        
    if allZerosForThirdWindow != True:
        percentageNumbersMask[84:  ] = True
    
    wholeNumbersToChange = np.sum(wholeNumbersMask)
    percentagesToChange  = np.sum(percentageNumbersMask)
    
    #print(wholeNumbersToChange.shape)
    
    wholeNumberNoise = np.random.normal(mean, stdDevWholeNumbers, size=wholeNumbersToChange)
    percentageNumberNoise = np.random.normal(mean, stdDevPercentages, size=percentagesToChange)
    
    #print(percentageNumberNoise)
    
    noisyPoint[wholeNumbersMask     ] = np.maximum(0, noisyPoint[wholeNumbersMask     ] + wholeNumberNoise)
    noisyPoint[percentageNumbersMask] = np.maximum(0, noisyPoint[percentageNumbersMask] + percentageNumberNoise)
    
    for i in range(len(noisyPoint)):
        if wholeNumbersMask[i] == True:
            noisyPoint[i] = math.ceil(noisyPoint[i])
            
    startAge = noisyPoint[0]
    
    noisyPoint[32] = startAge + 1
    noisyPoint[64] = startAge + 2
            
    noisyLabel = np.copy(label)
    
    #wholeNumbersLabelMask      = np.zeros_like(noisyLabel, dtype=bool)
    #percentageNumbersLabelMask = np.zeros_like(noisyLabel, dtype=bool)
    #
    #wholeNumbersLabelMask     [:4] = True
    #percentageNumbersLabelMask[4:] = True
    #
    #wholeNumbersLabelToChange      = np.sum(wholeNumbersLabelMask)
    #percentageNumbersLabelToChange = np.sum(percentageNumbersLabelMask)
    #
    #wholeNumbersLabelNoise      = np.random.normal(mean, stdDevWholeNumbers, size=wholeNumbersLabelToChange)
    #percentageNumbersLabelNoise = np.random.normal(mean, stdDevPercentages, size=percentageNumbersLabelToChange)
    #
    #noisyLabel[wholeNumbersLabelMask     ] = np.maximum(0, noisyLabel[wholeNumbersLabelMask     ] + wholeNumbersLabelNoise)
    #noisyLabel[percentageNumbersLabelMask] = np.maximum(0, noisyLabel[percentageNumbersLabelMask] + percentageNumbersLabelNoise)
    #
    #for i in range(len(noisyLabel)):
    #    if wholeNumbersLabelMask[i] == True:
    #        noisyLabel[i] = math.ceil(noisyLabel[i])
            
    noisyPointWindowed = noisyPoint.reshape(3, 32)
    
    return noisyPointWindowed, noisyLabel

In [ ]:
fullFeatures2d = fullFeaturesArray.reshape(fullFeaturesArray.shape[0], -1)

print(fullFeaturesArray.shape)

#0,1,4
noisyFeature, noisyLabel = addNoiseToPoints(fullFeatures2d[0], fullLabelsArray[0])

print(fullFeatures2d[0])
print("------------------------------------------------")
print(noisyFeature)

print()

print(fullLabelsArray[0])
print("------------------------------------------------")
print(noisyLabel)

In [ ]:
numClusters = 4

interpolatedFeatures = []
interpolatedLabels   = []

for i in range(len(binIndexes)):
    binCondition = (fullLabelsArrayBinned[:, 0] == binIndexes[i][0]) & (fullLabelsArrayBinned[:, 1] == binIndexes[i][1]) & (fullLabelsArrayBinned[:, 2] == binIndexes[i][2]) & (fullLabelsArrayBinned[:, 3] == binIndexes[i][3]) & (fullLabelsArrayBinned[:, 4] == binIndexes[i][4])
    
    matchingLabelIndices = np.where(binCondition)[0]
    
    matchingFeatures = fullFeaturesArray[matchingLabelIndices]
    matchingLabels   = fullLabelsArray  [matchingLabelIndices]
    
    #if matching features is less than number of clusters, we either need to reduce the size of the clusters
    #or if its size 1, add gaussian noise n times
    
    matchingFeatureLen   = matchingFeatures.shape[0]
    matchingFeatures2d   = matchingFeatures.reshape(matchingFeatureLen, -1)
    
    actualNumClusters = numClusters
    
    if len(matchingFeatures2d) < 4:
        actualNumClusters = 1
        
    neededPoints = 703 - len(matchingFeatures2d)
    
    if neededPoints == 0:
        continue
    
    #if a cluster has length 1, we should exclude it and only use those which we can interpolate new points from
    kmeans = KMeans(n_clusters=actualNumClusters, random_state=0)
    
    kmeans.fit(matchingFeatures2d)
    
    validClusterIndices = []
    
    for j in range(numClusters):
        clusterIndices = np.where(kmeans.labels_ == j)[0]
        numFeaturesInCluster = len(clusterIndices)
        
        #print(numFeaturesInCluster)
        
        if numFeaturesInCluster > 1:
            validClusterIndices.append(j)
            
    numValidClusters = len(validClusterIndices)
    
    if numValidClusters == 0:
        print(f"smote naive: {len(matchingFeatures2d)}, {neededPoints}")
        
        while neededPoints > 0:
            noisyPoint, noisyLabel = addNoiseToPoints(matchingFeatures2d[0], matchingLabels[0])
            
            interpolatedFeatures.append(noisyPoint.tolist())
            interpolatedLabels  .append(noisyLabel.tolist())
            
            neededPoints -=1
            
        print(len(interpolatedFeatures))
        print(len(interpolatedLabels))
             
    else:
        neededPointsPerCluster = math.floor(neededPoints / numValidClusters)
        
        print(f"smote unaive: {len(matchingFeatures2d)}, {neededPoints}, {neededPointsPerCluster}")
        
        newPointsInBin = 0
        
        for j in range(numValidClusters):
            clusterIndices            =  np.where(kmeans.labels_ == validClusterIndices[j])[0]
            matchingFeatures2dCluster = matchingFeatures2d[clusterIndices]
            matchingLabelsCluster     = matchingLabels    [clusterIndices]
            
            clusterLength = len(matchingFeatures2dCluster)
            pairsInCluster = (clusterLength * (clusterLength - 1))/2
            
            #print(f"cluster length: {clusterLength}")
            
            clusterRun = math.ceil(neededPointsPerCluster / pairsInCluster)
            
            for k in range(len(matchingFeatures2dCluster)):
                for l in range(k+1, len(matchingFeatures2dCluster)):
                    firstFeature = matchingFeatures2dCluster[k]
                    firstLabel   = matchingLabelsCluster    [k]
                    
                    secondFeature = matchingFeatures2dCluster[l]
                    secondLabel   = matchingLabelsCluster    [l]
                    
                    pointsNeeded = clusterRun
                    
                    stepSize = 1.0 / (pointsNeeded + 1.0)
                    
                    interpolatedFeature = []
                    
                    #also need to check if we've already gotten enough points for this cluster
                    #if either point has masked values, prefer gaussian noise of the one without over interpolating
                    #if both points have masked values, fill in all zeros
                    while pointsNeeded > 0 and newPointsInBin < neededPoints:
                        interpolatedFeatureWindow = []
                        
                        for m in range(len(firstFeature)):
                            if m == 32 or m == 64:
                                interpolatedFeature.append(interpolatedFeatureWindow)
                                
                                interpolatedFeatureWindow = []
                            
                            firstFeaturePoint  = firstFeature [m]
                            secondFeaturePoint = secondFeature[m]
                            
                            slope = max(secondFeaturePoint, firstFeaturePoint) - min(secondFeaturePoint, firstFeaturePoint)
                            
                            interpolatedPoint = min(secondFeaturePoint, firstFeaturePoint) + slope * pointsNeeded * stepSize
                            
                            if m <= 19 or (m >= 32 and m < 52) or (m >= 64 and m < 84):
                                interpolatedPoint = math.floor(interpolatedPoint)
                                
                            interpolatedFeatureWindow.append(interpolatedPoint)
                            
                            #print(f"{i}, {firstFeaturePoint}, {secondFeaturePoint}, {slope}, {pointsNeeded}, {stepSize}, {interpolatedPoint}")
                        
                        interpolatedFeature.append(interpolatedFeatureWindow)
                        
                        interpolatedLabel = []
                        
                        for m in range(len(firstLabel)):
                            firstLabelPoint  = firstLabel [m]
                            secondLabelPoint = secondLabel[m]
                            
                            slope = max(secondLabelPoint, firstLabelPoint) - min(secondLabelPoint, firstLabelPoint)
                            
                            interpolatedPoint = min(secondLabelPoint, firstLabelPoint) + slope * pointsNeeded * stepSize
                    
                            if m != 4:
                                interpolatedPoint = math.floor(interpolatedPoint)
                                
                            interpolatedLabel.append(interpolatedPoint)
                            
                            #print(f"{i}, {firstLabelPoint}, {secondLabelPoint}, {slope}, {interpolatedPoint}")
                            
                        pointsNeeded -= 1
                        newPointsInBin += 1
                        
                        interpolatedFeatures.append(interpolatedFeature)
                        interpolatedLabels  .append(interpolatedLabel  )
                        
                        interpolatedFeature = []
                        
        print(len(interpolatedFeatures))
        print(len(interpolatedLabels))

In [ ]:
for i in range(len(interpolatedFeatures)):
    if len(interpolatedFeatures[i]) != 3:
        print(f"BAD WINDOW AMOUNT: {i}, ACTUAL LENGTH: {len(interpolatedFeatures[i])}")
    for j in range(len(interpolatedFeatures[i])):
        if len(interpolatedFeatures[i][j])!= 32:
            print(f"BAD WINDOW LENGTH: {i}, {j}")
        

len(interpolatedFeatures[0][2])

In [ ]:
#interpolatedFeatures[32822]

In [ ]:
interpolatedFeaturesArray = np.array(interpolatedFeatures)
interpolatedLabelsArray   = np.array(interpolatedLabels)

In [ ]:
interpolatedFeaturesArray.shape

In [ ]:
concatenatedFeaturesArray = np.concatenate((fullFeaturesArray, interpolatedFeaturesArray), axis=0)
concatenatedLabelsArray   = np.concatenate((fullLabelsArray  , interpolatedLabelsArray  ), axis=0)

In [ ]:
concatenatedLabelsArray.shape

In [ ]:
concatenatedRunBins              = np.linspace(min(concatenatedLabelsArray[:,0]), max(concatenatedLabelsArray[:,0]), num=4)
concatenatedHomeRunsBins         = np.linspace(min(concatenatedLabelsArray[:,1]), max(concatenatedLabelsArray[:,1]), num=4)
concatenatedRbiBins              = np.linspace(min(concatenatedLabelsArray[:,2]), max(concatenatedLabelsArray[:,2]), num=4)
concatenatedStolenBaseBins       = np.linspace(min(concatenatedLabelsArray[:,3]), max(concatenatedLabelsArray[:,3]), num=4)
concatenatedOnBasePercentageBins = np.linspace(min(concatenatedLabelsArray[:,4]), max(concatenatedLabelsArray[:,4]), num=4)

concatenatedRunsBinned             = np.digitize(concatenatedLabelsArray[:,0], concatenatedRunBins, right=True)
concatenatedHomeRunsBinned         = np.digitize(concatenatedLabelsArray[:,1], concatenatedHomeRunsBins, right=True)
concatenatedRbisBinned             = np.digitize(concatenatedLabelsArray[:,2], concatenatedRbiBins, right=True)
concatenatedStolenBasesBinned      = np.digitize(concatenatedLabelsArray[:,3], concatenatedStolenBaseBins, right=True)
concatenatedOnBasePercentageBinned = np.digitize(concatenatedLabelsArray[:,4], concatenatedOnBasePercentageBins, right=True)

fullConcatenatedLabelsArrayBinned = np.column_stack((concatenatedRunsBinned, concatenatedHomeRunsBinned, concatenatedRbisBinned, concatenatedStolenBasesBinned, concatenatedOnBasePercentageBinned))

In [ ]:
concatedBinCount = {}

for label in fullConcatenatedLabelsArrayBinned:
    labelString = str(label[0]) + str(label[1]) + str(label[2]) + str(label[3]) + str(label[4])

    if labelString in concatedBinCount:
        concatedBinCount[labelString] = concatedBinCount[labelString] + 1
    else:
        concatedBinCount[labelString] = 1

In [ ]:
concatenatedSortedBinCount = dict(sorted(concatedBinCount.items(), key=lambda item: item[1], reverse=True))

In [ ]:
concatenatedBinOne   = {}
concatenatedBinTwo   = {}
concatenatedBinThree = {}
concatenatedBinFour  = {}

index = 1

for key in concatenatedSortedBinCount:
    if index <= 22:
        concatenatedBinOne[key] = concatenatedSortedBinCount[key]
    elif index > 22 and index <= 44:
        concatenatedBinTwo[key] = concatenatedSortedBinCount[key]
    elif index > 44 and index <= 66:
        concatenatedBinThree[key] = concatenatedSortedBinCount[key]
    else:
        concatenatedBinFour[key] = concatenatedSortedBinCount[key]

    index += 1

In [ ]:
concatenatedLabels = list(concatedBinCount.keys())
concatenatedCount  = list(concatedBinCount.values())

plt.figure(figsize=(16, 12))
bar = plt.bar(concatenatedLabels, concatenatedCount)

_ = plt.xticks(concatenatedLabels, rotation="vertical")
_ = plt.bar_label(bar, concatenatedCount)

In [ ]:
matchingFeatures = fullFeaturesArray[matchingLabelIndices]
matchingLabels   = fullLabelsArray  [matchingLabelIndices]

In [ ]:
matchingFeatureLen   = matchingFeatures.shape[0]
matchingFeatures2d   = matchingFeatures.reshape(matchingFeatureLen, -1)

In [ ]:
#matchingFeatures[0]

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=0)

In [ ]:
kmeans.fit(matchingFeatures2d)

In [ ]:
#kmeans.labels_

In [ ]:
distances = kmeans.transform(matchingFeatures2d)

In [ ]:
#distances

In [ ]:
#kmeans.cluster_centers_

In [ ]:
firstClusterIndices            =  np.where(kmeans.labels_ == 0)[0]
matchingFeatures2dFirstCluster = matchingFeatures2d[firstClusterIndices]
matchingLabelsFirstCluster     = matchingLabels[firstClusterIndices]

In [ ]:
neededPoints = 703 - len(matchingFeatures2d)

#print(neededPoints)

neededPointsPerCluster = math.floor(neededPoints / 4)

#print(neededPointsPerCluster)

clusterLength = len(matchingFeatures2dFirstCluster)
pairsInCluster = (clusterLength * (clusterLength - 1))/2

#print(clusterLength)
#print(pairsInCluster)

clusterRun = math.ceil(neededPointsPerCluster / pairsInCluster)

#print(clusterRun)

In [ ]:
firstFeature = matchingFeatures2dFirstCluster[0]
firstLabel   = matchingLabelsFirstCluster    [0]

secondFeature = matchingFeatures2dFirstCluster[1]
secondLabel   = matchingLabelsFirstCluster    [1]

steps = 1

#Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage

pointsNeeded = clusterRun

stepSize = 1.0 / (pointsNeeded + 1.0)

interpolatedFeatures = []
interpolatedLabels   = []

while pointsNeeded > 0:
    interpolatedFeature = []
    
    for i in range(len(firstFeature)):
        firstFeaturePoint  = firstFeature [i]
        secondFeaturePoint = secondFeature[i]
        
        slope = max(secondFeaturePoint, firstFeaturePoint) - min(secondFeaturePoint, firstFeaturePoint)
        
        interpolatedPoint = min(secondFeaturePoint, firstFeaturePoint) + slope * pointsNeeded * stepSize
        
        if i <= 19 or (i >= 32 and i < 52) or (i >= 64 and i < 84):
            interpolatedPoint = math.floor(interpolatedPoint)
            
        interpolatedFeature.append(interpolatedPoint)
        
        #print(f"{i}, {firstFeaturePoint}, {secondFeaturePoint}, {slope}, {pointsNeeded}, {stepSize}, {interpolatedPoint}")
    
    interpolatedLabel = []
    
    for i in range(len(firstLabel)):
        firstLabelPoint  = firstLabel[i]
        secondLabelPoint = secondLabel[i]
        
        slope = max(secondLabelPoint, firstLabelPoint) - min(secondLabelPoint, firstLabelPoint)
        
        interpolatedPoint = min(secondLabelPoint, firstLabelPoint) + slope * pointsNeeded * stepSize

        if i != 4:
            interpolatedPoint = math.floor(interpolatedPoint)
            
        interpolatedLabel.append(interpolatedPoint)
        
        #print(f"{i}, {firstLabelPoint}, {secondLabelPoint}, {slope}, {interpolatedPoint}")
        
    pointsNeeded -= 1
    
    interpolatedFeatures.append(interpolatedFeature)
    interpolatedLabels  .append(interpolatedLabel  )

In [ ]:
interpolatedFeatures[9]

In [ ]:
have

max number of data points in bin (~700)

m data points in b

4 clusters per bin

n data points in each cluster

700 - m = new data points needed/bin

700 - m / 4 = total data points needed per cluster

can choose steps in slope, or calculate that based on n(n-1)/2 total pairs



In [ ]:
neededPoints = 700 - len(matchingFeatures2d)

print(neededPoints)

neededPointsPerCluster = math.floor(neededPoints / 4)

print(neededPointsPerCluster)

clusterLength = len(matchingFeatures2dFirstCluster)
pairsInCluster = (clusterLength * (clusterLength - 1))/2

print(clusterLength)
print(pairsInCluster)

clusterRun = math.ceil(neededPointsPerCluster / pairsInCluster)

print(clusterRun)

In [ ]:
BREAK

In [ ]:
def getDistance(featureArrayOne, featureArrayTwo):
    totalDistance = 0.0
    for i in range(len(featureArrayOne)):
        for j in range(len(featureArrayOne[i])):
            featureOne = featureArrayOne[i][j]
            featureTwo = featureArrayTwo[i][j]

            totalDistance += math.sqrt(math.pow(featureTwo - featureOne, 2))

    return totalDistance

In [ ]:
we have

label, distances, ceenters

wee can find the center for the given input and create pooints in between each corresponding stat
then add that to the group

In [ ]:
minDistance = 1000000.0
minDistanceI = -1
minDistanceJ = -1

maxDistance = 0.0
maxDistanceI = -1
maxDistanceJ = -1

for i in range(len(matchingFeatures)):
    for j in range(i+1, len(matchingFeatures)):
        eucDistance = getDistance(matchingFeatures[i], matchingFeatures[j])

        if eucDistance < minDistance:
            minDistance = eucDistance
            minDistanceI = i
            minDistanceJ = j

        if eucDistance > maxDistance:
            maxDistance = eucDistance
            maxDistanceI = i
            maxDistanceJ = j
            

print(f"{minDistance}, {minDistanceI}, {minDistanceJ}")
print(f"{maxDistance}, {maxDistanceI}, {maxDistanceJ}")

In [ ]:
#REPLACE MASK WITH ALL MINUS 1

matchingFeatures[411]

In [ ]:
matchingFeatures[420]

In [ ]:
labels = list(sortedBinCount.keys())
count  = list(sortedBinCount.values())

plt.figure(figsize=(16, 12))
plt.bar(labels, count)

_ = plt.xticks(labels, rotation="vertical")

In [ ]:
binCount["21132"]

In [ ]:
fullLabelsArrayBinned[1]

In [ ]:
logColumns     = df.columns[3:]
df[logColumns] = np.log(df[logColumns] + 1)

In [ ]:
df['rbis'].hist(bins=50)
plt.title('Histogram of col1')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
transformedDf = df.copy(deep=True)

In [ ]:
# yeoJohnTransformer = PowerTransformer(standardize=True)

# transformedTrainDf['stolenBases'] = pd.DataFrame(yeoJohnTransformer.fit_transform(df['stolenBases'].values.reshape(-1,1)))

In [ ]:
# Applying Quantile Transformation to follow a normal distribution
quantile_transformer = QuantileTransformer(output_distribution='normal', random_state=0)
transformedDf['stolenBases'] = quantile_transformer.fit_transform(transformedDf['stolenBases'].values.reshape(-1, 1)).flatten()

In [ ]:
transformedDf['stolenBases'].hist(bins=50)
plt.title('Histogram of col1')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
transformedTrainDf

In [ ]:
numericalDf   = df.select_dtypes(include=np.number)
categoricalDf = df.select_dtypes(exclude=np.number)

dfMean = numericalDf.mean()
dfStd  = numericalDf.std ()

dfNormalized = (numericalDf - dfMean) / dfStd

dfNormalized = pd.concat([categoricalDf, dfNormalized], axis=1)

In [ ]:
labels